# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

I build one feature table per content item using two sources:
daily performance aggregates (impression/click momentum,
position volatility) and 90-day query-mix signals. Missing
categorical/ratio features are filled with 0 after the join,
since a missing query-mix value means the page had no matching
query-level data, not an unknown value.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [27]:
%pip -q install duckdb huggingface_hub

import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients':       f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content':       f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':        f"read_parquet('{REL}/fact_content_daily_performance_sample.parquet')",  # sample instead of full
    'fact_query_90d':    f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

# Rebuild the momentum features using the LIGHT sample table
features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL 30 DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL 60 DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

print(f'features: {len(features):,} rows | qsignals: {len(qsignals):,} rows')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features: 0 rows | qsignals: 133,852 rows


In [37]:
span = con.sql(f"""
    SELECT MIN(report_date) AS min_d, MAX(report_date) AS max_d,
           DATEDIFF('day', MIN(report_date), MAX(report_date)) AS total_days
    FROM {TABLES['fact_daily']}
""").df()
print(span)

       min_d      max_d  total_days
0 2026-06-01 2026-06-30          29


In [38]:
half_window = span['total_days'].iloc[0] // 2   # ≈ 14 بناءً على 29 يوم متاحين

features = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']}),
    windowed AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30,
               SUM(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_clicks ELSE 0 END)      AS clk_last30,
               AVG(CASE WHEN f.report_date >  b.end_d - INTERVAL {half_window} DAY THEN f.gsc_avg_position END)       AS pos_last30
        FROM {TABLES['fact_daily']} f, bounds b
        WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
        GROUP BY 1, 2
        HAVING imp_prev30 >= 100
    )
    SELECT * FROM windowed
""").df()

print(f'features: {len(features):,} rows')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

features: 77,155 rows


In [39]:
for thresh in [0, 5, 10, 20, 50, 100]:
    n = con.sql(f"""
        WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']})
        SELECT COUNT(*) AS n FROM (
            SELECT f.client_hash_id, f.content_hash_id,
                   SUM(CASE WHEN f.report_date <= b.end_d - INTERVAL {half_window} DAY THEN f.gsc_impressions ELSE 0 END) AS imp_prev30
            FROM {TABLES['fact_daily']} f, bounds b
            WHERE f.report_date > b.end_d - INTERVAL {half_window * 2} DAY
            GROUP BY 1, 2
            HAVING imp_prev30 >= {thresh}
        )
    """).df()['n'].iloc[0]
    print(f"threshold={thresh}: {n:,} rows")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

threshold=0: 409,205 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

threshold=5: 150,535 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

threshold=20: 121,670 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

threshold=50: 97,567 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

threshold=100: 77,155 rows


In [40]:
qsignals = con.sql(f"""
    SELECT content_hash_id,
           ANY_VALUE(content_visible_query_count)     AS visible_queries,
           ANY_VALUE(rare_impressions_share)          AS rare_share,
           ANY_VALUE(anonymized_impressions_share)    AS anon_share,
           MAX(impressions_90d)                       AS top_query_impressions,
           SUM(impressions_90d)                       AS kept_impressions
    FROM {TABLES['fact_query_90d']}
    GROUP BY content_hash_id
""").df()

print(f'features: {len(features):,} rows | qsignals: {len(qsignals):,} rows')

features: 77,155 rows | qsignals: 133,852 rows


In [41]:
data = features.merge(qsignals, on='content_hash_id', how='left')

volatility = con.sql(f"""
    WITH bounds AS (SELECT MAX(report_date) AS end_d FROM {TABLES['fact_daily']})
    SELECT f.content_hash_id, STDDEV(f.gsc_avg_position) AS position_volatility
    FROM {TABLES['fact_daily']} f, bounds b
    WHERE f.report_date > b.end_d - INTERVAL 90 DAY
    GROUP BY 1
""").df()
data = data.merge(volatility, on='content_hash_id', how='left')

data['top_query_share'] = data['top_query_impressions'] / data['kept_impressions']

fill_cols = ['visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility']
data[fill_cols] = data[fill_cols].fillna(0)

data.head()

,client_hash_id,content_hash_id,imp_last30,imp_prev30,clk_last30,pos_last30,visible_queries,rare_share,anon_share,top_query_impressions,kept_impressions,position_volatility,top_query_share
0,client_65de48885f4ef01b,content_3f21e6f5f245a6e5,157.0,140.0,0.0,44.024782,0.0,0.0,0.0,NaN,NaN,9.524016,0.0
1,client_65de48885f4ef01b,content_7a5910945054b16e,474.0,399.0,3.0,6.737103,0.0,0.0,0.0,NaN,NaN,2.956669,0.0
2,client_65de48885f4ef01b,content_7745c5cc15c70a7b,116.0,209.0,0.0,5.138693,0.0,0.0,0.0,NaN,NaN,1.912190,0.0
3,client_65de48885f4ef01b,content_8c021591c0def13d,1820.0,2110.0,4.0,7.342609,0.0,0.0,0.0,NaN,NaN,1.062332,0.0
4,client_65de48885f4ef01b,content_8aa6a4350440755e,0.0,117.0,0.0,NaN,0.0,0.0,0.0,NaN,NaN,4.857252,0.0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

## 2. Feature notes (meaning, missing, categorical, available-when?)
For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.

Feature | Meaning | Missing handling | Available before prediction window?
---|---|---|---
imp_prev30 | Impressions in the [half_window]-day period BEFORE the outcome window | dropped (required by HAVING clause) | Yes
visible_queries | Count of distinct queries a page ranks for | filled 0 | Yes
rare_share | Share of impressions from rare/long-tail queries | filled 0 | Yes
anon_share | Share of impressions from anonymized queries | filled 0 | Yes
top_query_share | Concentration on the single top query | filled 0 | Yes
position_volatility | Std dev of daily avg. search position | filled 0 | Yes

All six features are computed only from data available strictly before the outcome window used to define the label — none of them use imp_last30 or any data from the prediction window itself.

**Note on window size:** the sample table only covers days total, not the 60 days the original design assumed. Windows were rebuilt as two roughly equal halves of the available range instead of fixed 30/30-day periods, and the HAVING threshold was re-checked against the actual impression distribution rather than kept at an arbitrary 100.

In [42]:
feature_cols = ['imp_prev30', 'visible_queries', 'rare_share',
                 'anon_share', 'top_query_share', 'position_volatility']

assert 'imp_last30' not in feature_cols, "Leakage: outcome-window column in features"
assert 'is_declining' not in feature_cols, "Leakage: label itself in features"
print("No outcome-window columns present in feature_cols:", feature_cols)

No outcome-window columns present in feature_cols: ['imp_prev30', 'visible_queries', 'rare_share', 'anon_share', 'top_query_share', 'position_volatility']


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Two leakage risks were tested directly. First, whether the label
window overlaps the feature window (it doesn't — features use
prev-30/90-day windows, the label uses the last-30-day outcome).
Second, and more importantly, whether a random train/test split
lets the model "see" a client's other pages during training and
then get tested on that same client — this inflates results
without the model learning anything that generalizes. I compared
a random split against a GroupShuffleSplit grouped by client_hash_id.

In [43]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

data['is_declining'] = (data['imp_last30'] < 0.8 * data['imp_prev30']).astype(int)
model_data = data.dropna(subset=feature_cols).reset_index(drop=True)
X, y = model_data[feature_cols], model_data['is_declining']
groups = model_data['client_hash_id']

X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

clf_leaky = RandomForestClassifier(n_estimators=200, random_state=42)
clf_leaky.fit(X_tr, y_tr)
pred_leaky = clf_leaky.predict(X_te)

clients_train_leaky = set(model_data.loc[X_tr.index, 'client_hash_id'])
clients_test_leaky  = set(model_data.loc[X_te.index, 'client_hash_id'])
overlap_leaky = clients_train_leaky & clients_test_leaky

print("=== Leaky (random) split ===")
print(f"Clients in train: {len(clients_train_leaky)} | Clients in test: {len(clients_test_leaky)}")
print(f"Clients appearing in BOTH train and test: {len(overlap_leaky)}")
print(classification_report(y_te, pred_leaky, digits=3))

=== Leaky (random) split ===
Clients in train: 45 | Clients in test: 45
Clients appearing in BOTH train and test: 45
              precision    recall  f1-score   support

           0      0.636     0.673     0.654      9676
           1      0.650     0.612     0.631      9613

    accuracy                          0.643     19289
   macro avg      0.643     0.642     0.642     19289
weighted avg      0.643     0.643     0.642     19289



In [44]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_tr_g, X_te_g = X.iloc[train_idx], X.iloc[test_idx]
y_tr_g, y_te_g = y.iloc[train_idx], y.iloc[test_idx]

clf_grouped = RandomForestClassifier(n_estimators=200, random_state=42)
clf_grouped.fit(X_tr_g, y_tr_g)
pred_grouped = clf_grouped.predict(X_te_g)

clients_train_g = set(groups.iloc[train_idx])
clients_test_g  = set(groups.iloc[test_idx])
overlap_g = clients_train_g & clients_test_g

print("=== Clean (grouped-by-client) split ===")
print(f"Clients in train: {len(clients_train_g)} | Clients in test: {len(clients_test_g)}")
print(f"Clients appearing in BOTH train and test: {len(overlap_g)}")
print(classification_report(y_te_g, pred_grouped, digits=3))

=== Clean (grouped-by-client) split ===
Clients in train: 33 | Clients in test: 12
Clients appearing in BOTH train and test: 0
              precision    recall  f1-score   support

           0      0.718     0.505     0.593     15961
           1      0.566     0.765     0.651     13460

    accuracy                          0.624     29421
   macro avg      0.642     0.635     0.622     29421
weighted avg      0.649     0.624     0.620     29421



The clean grouped-by-client split achieved an F1 score of **0.622**, with **33 clients in the training set and 12 clients in the test set**. There were **0 clients overlapping** between train and test.

This grouped split provides a more honest estimate of model performance because it prevents the model from seeing the same clients during both training and testing. This reduces the risk of **data leakage** and ensures that the model is evaluated on clients it has never trained on. Therefore, the **GroupShuffleSplit result (F1 = 0.622)** is the number I am reporting as the honest estimate of how the model will perform when scoring pages for unseen clients.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

## 4. What I excluded and why

Excluded field | Why
---|---
client_hash_id, content_hash_id | Identifiers, not signal — using them lets the model memorize rows/clients instead of learning a generalizable pattern
clk_last30, pos_last30 | Both fall inside the same outcome window used to build the label — using them as features would leak the outcome into the input
top_query_impressions, kept_impressions (raw) | Kept only as intermediates to build top_query_share; as raw counts they mostly reflect page size/popularity rather than a comparable signal
Any raw client name, URL, or query text | Privacy — no client names, URLs, or private queries appear anywhere in the notebook
report_date | Used only to define window boundaries, never joined into the feature table itself

In [45]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

excluded_cols = ['client_hash_id', 'content_hash_id', 'clk_last30', 'pos_last30',
                  'top_query_impressions', 'kept_impressions', 'report_date']
assert not set(excluded_cols) & set(feature_cols), "Excluded field leaked back into feature_cols"
print("Confirmed: none of the excluded fields are in feature_cols.")

Confirmed: none of the excluded fields are in feature_cols.


Self-check
Before you submit, confirm each line honestly:

- Every section above is filled — markdown thinking AND the code that backs it
- The notebook runs top to bottom with no errors (Runtime → Run all)
- No client names, URLs, or private queries anywhere
- My claims use careful words: observed, measured, directional, decision-support
- Committed to my repo under work/notebooks/ — then submit your repo URL on the card. Done.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.